# StageBridge
**Cross-modal spatial–snRNA-seq bridge for lung precursor-to-LUAD progression**

---

## Prerequisites

Before running this notebook you must have:

1. Activated the micromamba environment:
   ```bash
   micromamba activate stagebridge
   ```
2. Converted all snRNA files to h5ad:
   ```bash
   python scripts/run_snrna_pipeline.py
   ```
3. Expanded and converted all spatial samples:
   ```bash
   python scripts/run_spatial_pipeline.py
   ```

This notebook reads from:
- `$STAGEBRIDGE_DATA_ROOT/processed/anndata/snrna_merged.h5ad`
- `$STAGEBRIDGE_DATA_ROOT/processed/anndata/spatial_merged.h5ad`

It writes outputs to:
- `./outputs/figures/`  (repo-local)


## 0 — Imports & Configuration

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # change to 'inline' for interactive display
import matplotlib.pyplot as plt
import anndata
import scanpy as sc

# ── GPU check ─────────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU : {GPU_NAME}  ({GPU_MEM_GB:.1f} GB VRAM)")
        print(f"CUDA: {torch.version.cuda}")
        USE_GPU = True
    else:
        print("PyTorch installed but no CUDA GPU found — running on CPU.")
        USE_GPU = False
except ImportError:
    print("PyTorch not installed.")
    USE_GPU = False

# ── Optional packages ─────────────────────────────────────────────────────
try:
    import squidpy as sq
    SQUIDPY = True
    print(f"squidpy {sq.__version__}")
except ImportError:
    SQUIDPY = False
    print("squidpy not installed — spatial stats steps will be skipped")

try:
    import harmonypy
    HARMONY = True
    print("harmonypy available")
except ImportError:
    HARMONY = False
    print("harmonypy not installed — batch correction step will be skipped")

try:
    import scvi
    SCVI = True
    print(f"scvi-tools {scvi.__version__}")
    if USE_GPU:
        scvi.settings.dl_num_workers = 4   # data loader workers
        # scvi-tools auto-detects GPU; confirm with:
        print(f"  scvi accelerator: {'gpu' if USE_GPU else 'cpu'}")
except ImportError:
    SCVI = False
    print("scvi-tools not installed — scVI/scANVI steps will be skipped")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1

# ── Repo root on path (if not installed as a package) ─────────────────────
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from stagebridge.logging_utils import configure_root_logger
configure_root_logger()

from stagebridge import config
from stagebridge.preprocessing.harmonize import (
    intersect_genes,
    normalize_log1p,
    select_hvg,
    pca_fit_transform_snrna,
    pca_transform_spatial,
    run_harmony,
    run_umap,
)

FIGURES_DIR = REPO_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nRepo root : {REPO_ROOT}")
print(f"Figures   : {FIGURES_DIR}")

PyTorch not installed.
squidpy not installed — spatial stats steps will be skipped
harmonypy not installed — batch correction step will be skipped
scvi-tools not installed — scVI/scANVI steps will be skipped

Repo root : /home/ajbook/projects/StageBridge
Figures   : /home/ajbook/projects/StageBridge/outputs/figures


In [2]:
# ── Resolve data root ──────────────────────────────────────────────────────
try:
    DATA_ROOT = config.get_data_root()
    print(f"Data root : {DATA_ROOT}")
except ValueError as e:
    print(f"[ERROR] {e}")
    raise SystemExit(1)

SNRNA_H5AD   = config.snrna_merged_h5ad()
SPATIAL_H5AD = config.spatial_merged_h5ad()

missing = [p for p in (SNRNA_H5AD, SPATIAL_H5AD) if not p.exists()]
if missing:
    lines = ["\n[ERROR] Processed h5ad file(s) not found:"]
    for p in missing:
        lines.append(f"  - {p}")
    lines += [
        "\nRun the conversion scripts first:",
        "  python scripts/run_snrna_pipeline.py",
        "  python scripts/run_spatial_pipeline.py",
    ]
    raise FileNotFoundError("\n".join(lines))

print(f"snRNA h5ad  : {SNRNA_H5AD}")
print(f"Spatial h5ad: {SPATIAL_H5AD}")

Data root : /mnt/e/StageBridge_data


FileNotFoundError: 
[ERROR] Processed h5ad file(s) not found:
  - /mnt/e/StageBridge_data/processed/anndata/snrna_merged.h5ad
  - /mnt/e/StageBridge_data/processed/anndata/spatial_merged.h5ad

Run the conversion scripts first:
  python scripts/run_snrna_pipeline.py
  python scripts/run_spatial_pipeline.py

## 1 — Load Processed Data

In [ ]:
print("Loading snRNA merged h5ad ...")
adata_rna = anndata.read_h5ad(SNRNA_H5AD)
print(f"  snRNA  shape : {adata_rna.shape}")
print(f"  obs cols     : {list(adata_rna.obs.columns)}")
print(f"  layers       : {list(adata_rna.layers.keys())}")

print()
print("Loading spatial merged h5ad ...")
adata_sp = anndata.read_h5ad(SPATIAL_H5AD)
print(f"  Spatial shape: {adata_sp.shape}")
print(f"  obs cols     : {list(adata_sp.obs.columns)}")
print(f"  obsm keys    : {list(adata_sp.obsm.keys())}")
print(f"  has spatial  : {'spatial' in adata_sp.obsm}")

## 2 — Dataset Summary

In [ ]:
def summarise(adata, label):
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"{'─'*55}")
    print(f"  Cells/spots : {adata.n_obs:,}")
    print(f"  Genes       : {adata.n_vars:,}")
    if "sample_id" in adata.obs.columns:
        print(f"  Samples     : {adata.obs['sample_id'].nunique()}")
    if "patient_id" in adata.obs.columns:
        patients = sorted(adata.obs["patient_id"].unique())
        print(f"  Patients    : {len(patients)}  ({', '.join(patients)})")
    if "stage" in adata.obs.columns:
        stage_counts = adata.obs["stage"].value_counts()
        print("  Stages:")
        for stage, n in stage_counts.items():
            print(f"    {stage:<15} {n:>8,}")
    if "spatial" in adata.obsm:
        c = adata.obsm["spatial"]
        print(f"  Spatial     : shape={c.shape}  "
              f"x=[{c[:,0].min():.0f},{c[:,0].max():.0f}]  "
              f"y=[{c[:,1].min():.0f},{c[:,1].max():.0f}]")

summarise(adata_rna, "snRNA-seq (GSE308103)")
summarise(adata_sp,  "Spatial Visium (GSE307534)")

## 3 — Gene Intersection

In [ ]:
print(f"Before: snRNA {adata_rna.n_vars:,} genes | Spatial {adata_sp.n_vars:,} genes")
adata_rna_int, adata_sp_int = intersect_genes(adata_rna, adata_sp)
print(f"After : {adata_rna_int.n_vars:,} shared genes")

## 4 — Normalisation (log1p via scanpy)

In [ ]:
layer_in = "counts" if "counts" in adata_rna_int.layers else None

print("Normalising snRNA ...")
normalize_log1p(adata_rna_int, layer_in=layer_in, layer_out="log1p")

sp_layer_in = "counts" if "counts" in adata_sp_int.layers else None
print("Normalising spatial ...")
normalize_log1p(adata_sp_int, layer_in=sp_layer_in, layer_out="log1p")

print("Layers — snRNA  :", list(adata_rna_int.layers.keys()))
print("Layers — Spatial:", list(adata_sp_int.layers.keys()))

## 5 — Highly Variable Gene Selection (scanpy)

In [ ]:
N_HVG = 2000

# seurat_v3 expects raw counts; use counts layer if available
hvg_layer = "counts" if "counts" in adata_rna_int.layers else "log1p"
hvg_flavor = "seurat_v3" if hvg_layer == "counts" else "seurat"

hvg_genes = select_hvg(adata_rna_int, n_hvg=N_HVG, layer=hvg_layer, flavor=hvg_flavor)
print(f"Selected {len(hvg_genes)} HVGs (flavor={hvg_flavor})")
print(f"Example HVGs: {hvg_genes[:10]}")

# Subset both modalities to HVGs present in both
common_hvg = [g for g in hvg_genes if g in set(adata_sp_int.var_names)]
print(f"{len(common_hvg)} of {len(hvg_genes)} HVGs present in spatial data")

adata_rna_hvg = adata_rna_int[:, common_hvg].copy()
adata_sp_hvg  = adata_sp_int[:,  common_hvg].copy()

## 6 — PCA

In [ ]:
N_PCA = 64

print(f"Fitting PCA ({N_PCA} components) on snRNA "
      f"({adata_rna_hvg.n_obs:,} cells × {adata_rna_hvg.n_vars:,} HVGs) ...")
pca_model = pca_fit_transform_snrna(adata_rna_hvg, n_components=N_PCA, use_layer="log1p")

print(f"Projecting spatial ({adata_sp_hvg.n_obs:,} spots) into snRNA PCA space ...")
pca_transform_spatial(adata_sp_hvg, pca_model, use_layer="log1p")

print(f"snRNA  PCA: {adata_rna_hvg.obsm['X_pca'].shape}")
print(f"Spatial PCA: {adata_sp_hvg.obsm['X_pca'].shape}")

## 7 — Harmony Batch Correction (across patients)

In [ ]:
if HARMONY and "patient_id" in adata_rna_hvg.obs.columns:
    run_harmony(adata_rna_hvg, batch_key="patient_id")
    print(f"Harmony embedding: {adata_rna_hvg.obsm['X_pca_harmony'].shape}")
    HARMONY_DONE = True
else:
    print("Skipping Harmony (harmonypy not installed or no patient_id column).")
    HARMONY_DONE = False

# The basis for downstream steps
EMBED_KEY = "X_pca_harmony" if HARMONY_DONE else "X_pca"

## 8 — UMAP (via scanpy)

In [ ]:
print(f"Computing UMAP for snRNA (basis='{EMBED_KEY}') ...")
run_umap(adata_rna_hvg, basis=EMBED_KEY, n_neighbors=15)
print(f"UMAP done: {adata_rna_hvg.obsm['X_umap'].shape}")

## 9 — Spatial Neighbourhood Graph (squidpy)

In [ ]:
if SQUIDPY and "spatial" in adata_sp_hvg.obsm:
    print("Building spatial neighbourhood graph (squidpy) ...")
    sq.gr.spatial_neighbors(adata_sp_hvg, coord_type="generic", n_neighs=6)
    print("obsp keys:", list(adata_sp_hvg.obsp.keys()))

    # Spatial autocorrelation (Moran's I) on the first PCA component
    print("Computing Moran's I on PC1 ...")
    adata_sp_hvg.obs["PC1"] = adata_sp_hvg.obsm["X_pca"][:, 0]
    sq.gr.spatial_autocorr(
        adata_sp_hvg,
        attr="obs",
        genes=["PC1"],
        mode="moran",
        n_perms=100,
        n_jobs=1,
    )
    moran_df = adata_sp_hvg.uns["moranI"]
    print(moran_df)
else:
    print("Skipping squidpy spatial graph (squidpy not installed or no spatial coords).")

## 10 — Visualisation

In [ ]:
# ── 10a: UMAP coloured by stage ────────────────────────────────────────────
stage_col = "stage" if "stage" in adata_rna_hvg.obs.columns else None

unique_stages = sorted(adata_rna_hvg.obs[stage_col].unique()) if stage_col else []
palette = plt.cm.tab10.colors
stage2color = {s: palette[i % len(palette)] for i, s in enumerate(unique_stages)}

fig, ax = plt.subplots(figsize=(7, 6))
umap = adata_rna_hvg.obsm["X_umap"]
if stage_col:
    stages = adata_rna_hvg.obs[stage_col].values
    for stage in unique_stages:
        m = stages == stage
        ax.scatter(umap[m, 0], umap[m, 1],
                   c=[stage2color[stage]], s=3, alpha=0.4,
                   label=stage, rasterized=True)
    ax.legend(markerscale=4, fontsize=9)
else:
    ax.scatter(umap[:, 0], umap[:, 1], s=3, alpha=0.4, rasterized=True)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.set_title(f"snRNA UMAP — {adata_rna_hvg.n_obs:,} cells  (Harmony+)" if HARMONY_DONE
             else f"snRNA UMAP — {adata_rna_hvg.n_obs:,} cells")
fig.tight_layout()
out = FIGURES_DIR / "umap_snrna_stage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.close(fig)

In [ ]:
# ── 10b: PCA scatter — spatial in snRNA space ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def scatter_pca(ax, adata, col, title, alpha=0.3):
    X = adata.obsm["X_pca"]
    if col and col in adata.obs.columns:
        stages = adata.obs[col].values
        for stage in unique_stages:
            m = stages == stage
            if m.any():
                ax.scatter(X[m, 0], X[m, 1], c=[stage2color[stage]],
                           s=4, alpha=alpha, label=stage, rasterized=True)
        ax.legend(markerscale=3, fontsize=8)
    else:
        ax.scatter(X[:, 0], X[:, 1], s=4, alpha=alpha, rasterized=True)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_title(title, fontsize=11)

scatter_pca(axes[0], adata_rna_hvg, stage_col, "snRNA PCA (PC1 vs PC2)")
scatter_pca(axes[1], adata_sp_hvg,  stage_col, "Spatial projected into snRNA PCA", alpha=0.5)
fig.suptitle("StageBridge — PCA Embedding (log1p, HVGs)", fontsize=13, y=1.01)
fig.tight_layout()
out = FIGURES_DIR / "pca_scatter.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.close(fig)

In [ ]:
# ── 10c: Stage distribution ────────────────────────────────────────────────
if stage_col:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, adata, label in [
        (axes[0], adata_rna_hvg, "snRNA"),
        (axes[1], adata_sp_hvg,  "Spatial"),
    ]:
        if stage_col in adata.obs.columns:
            counts = adata.obs[stage_col].value_counts().sort_index()
            ax.bar(counts.index, counts.values,
                   color=[stage2color.get(s, "gray") for s in counts.index])
            ax.set_title(f"{label} — Stage Distribution")
            ax.set_ylabel("# Cells / Spots")
            for t in ax.get_xticklabels(): t.set_rotation(30)
    fig.tight_layout()
    out = FIGURES_DIR / "stage_distribution.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)

In [ ]:
# ── 10d: Spatial tissue map (squidpy) ─────────────────────────────────────
if SQUIDPY and "spatial" in adata_sp_hvg.obsm:
    # Pick first sample
    first_id = adata_sp_hvg.obs["sample_id"].iloc[0] if "sample_id" in adata_sp_hvg.obs.columns else None
    sub = adata_sp_hvg[adata_sp_hvg.obs["sample_id"] == first_id] if first_id else adata_sp_hvg

    # squidpy spatial scatter
    if stage_col and stage_col in sub.obs.columns:
        color_arg = stage_col
    else:
        sub.obs["PC1"] = sub.obsm["X_pca"][:, 0]
        color_arg = "PC1"

    fig, ax = plt.subplots(figsize=(6, 6))
    sq.pl.spatial_scatter(
        sub,
        color=color_arg,
        ax=ax,
        size=1.2,
        title=f"Spatial tissue — {first_id or 'all'}",
    )
    out = FIGURES_DIR / f"spatial_tissue_{first_id or 'all'}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)
elif "spatial" in adata_sp_hvg.obsm:
    # Manual fallback plot
    coords = adata_sp_hvg.obsm["spatial"]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(coords[:, 0], coords[:, 1], s=6, alpha=0.6, rasterized=True)
    ax.set_title("Spatial tissue map (all samples)")
    ax.set_aspect("equal"); ax.invert_yaxis()
    out = FIGURES_DIR / "spatial_tissue_all.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)
else:
    print("No spatial coordinates available.")

## 11 — scVI / scANVI (optional, requires scvi-tools)

In [ ]:
if SCVI:
    import torch

    # ── scVI setup ────────────────────────────────────────────────────────
    # scVI expects raw integer counts in adata.X.
    adata_scvi = adata_rna_hvg.copy()
    if "counts" in adata_scvi.layers:
        adata_scvi.X = adata_scvi.layers["counts"]

    batch_key  = "patient_id" if "patient_id" in adata_scvi.obs.columns else None
    labels_key = "stage"      if "stage"      in adata_scvi.obs.columns else None

    scvi.model.SCVI.setup_anndata(
        adata_scvi,
        layer=None,           # X is already raw counts
        batch_key=batch_key,
        labels_key=labels_key,
    )
    model_scvi = scvi.model.SCVI(
        adata_scvi,
        n_layers=2,
        n_latent=32,
        n_hidden=256,
        gene_likelihood="nb",    # negative binomial — appropriate for UMI counts
        dispersion="gene-batch", # per-gene, per-batch dispersion
    )
    print(model_scvi)

    # ── scANVI setup (semi-supervised with stage labels) ──────────────────
    # scANVI learns a latent space that is batch-corrected AND stage-aware.
    # Central to StageBridge's staging objective.
    if labels_key:
        scvi.model.SCANVI.setup_anndata(
            adata_scvi,
            layer=None,
            batch_key=batch_key,
            labels_key=labels_key,
            unlabeled_category="Unknown",
        )
        model_scanvi = scvi.model.SCANVI(
            adata_scvi,
            n_layers=2,
            n_latent=32,
            n_hidden=256,
            gene_likelihood="nb",
        )
        print(model_scanvi)

    # ── Model output paths (external data root) ───────────────────────────
    SCVI_DIR   = config.scvi_model_dir()
    SCANVI_DIR = config.scanvi_model_dir()

    # ── Training (GPU) ─────────────────────────────────────────────────────
    print()
    if USE_GPU:
        print(f"GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB) — training will use CUDA.")
        print()
        print("# ── Run these cells to train: ─────────────────────────────────")
        print(f"model_scvi.train(max_epochs=400, accelerator='gpu', devices=1)")
        print(f"adata_rna_hvg.obsm['X_scVI'] = model_scvi.get_latent_representation()")
        print(f"model_scvi.save('{SCVI_DIR}', overwrite=True)")
        print()
        if labels_key:
            print(f"# Recommended: initialise scANVI from the trained scVI model")
            print(f"model_scanvi = scvi.model.SCANVI.from_scvi_model(model_scvi, labels_key='stage')")
            print(f"model_scanvi.train(max_epochs=20, accelerator='gpu', devices=1)")
            print(f"adata_rna_hvg.obsm['X_scANVI'] = model_scanvi.get_latent_representation()")
            print(f"stage_pred = model_scanvi.predict()   # predicted stage labels")
            print(f"model_scanvi.save('{SCANVI_DIR}', overwrite=True)")
    else:
        print("No GPU detected — scVI training on CPU will be very slow for >10k cells.")
        print("Activate the stagebridge env in a terminal with GPU access and run:")
        print("  jupyter lab StageBridge.ipynb")
else:
    print("scvi-tools not installed — skipping scVI/scANVI setup.")
    print("Install with:  pip install scvi-tools")

## 12 — Summary

In [ ]:
print("=" * 65)
print("  StageBridge preprocessing summary")
print("=" * 65)
print(f"  snRNA  cells × HVGs  : {adata_rna_hvg.shape}")
print(f"  Spatial spots × HVGs : {adata_sp_hvg.shape}")
print(f"  PCA dims             : {adata_rna_hvg.obsm['X_pca'].shape[1]}")
print(f"  Harmony corrected    : {HARMONY_DONE}")
print(f"  UMAP computed        : {'X_umap' in adata_rna_hvg.obsm}")
print(f"  Spatial graph        : {'connectivities' in adata_sp_hvg.obsp}")
print(f"  Figures saved to     : {FIGURES_DIR}")
print("=" * 65)
print()
print("Next steps:")
print("  • Train scVI/scANVI on snRNA (cell 11 above)")
print("  • Run Tangram spatial mapping (stagebridge/models/stagebridge.py)")
print("  • Add cell2location deconvolution")

---

## 13 — Build Latent Representation (HLCA-aligned or PCA fallback)

The `build_latent` helper adds `adata.obsm["X_hlca"]` which is the latent space used for all downstream training.  
If HLCA reference is unavailable it silently falls back to PCA — **training is never blocked**.


In [ ]:
from stagebridge.preprocessing.latent import build_latent, latent_summary
from stagebridge.preprocessing.harmonize import ensure_required_obs_fields

# ── Ensure required obs metadata fields are populated ─────────────────────
ensure_required_obs_fields(adata_rna_hvg)

# ── Try to load HLCA reference (graceful: None if file not found) ──────────
HLCA_REFERENCE = None
try:
    from stagebridge.io.hlca import load_hlca_reference, hlca_reference_h5ad
    _hlca_path = hlca_reference_h5ad()
    if _hlca_path.exists():
        HLCA_REFERENCE = load_hlca_reference(h5ad_path=_hlca_path)
        print(f"HLCA reference loaded from {_hlca_path}")
    else:
        print(f"HLCA reference not found at {_hlca_path} — using PCA-only alignment.")
except Exception as _e:
    print(f"HLCA load skipped ({_e}) — using PCA-only alignment.")

# ── Build latent embedding ─────────────────────────────────────────────────
LATENT_KEY   = "X_hlca"
N_LATENT_DIM = 64

build_latent(
    adata_rna_hvg,
    method="hlca",
    n_components=N_LATENT_DIM,
    output_key=LATENT_KEY,
    pca_layer="log1p",
    hlca_reference=HLCA_REFERENCE,
)

info = latent_summary(adata_rna_hvg, output_key=LATENT_KEY)
print(f"\nLatent embedding '{LATENT_KEY}':")
for k, v in info.items():
    print(f"  {k:<12}: {v}")

## 14 — Donor-Held-Out Splits

Five-fold (or three-fold for small cohorts) donor-held-out CV splits.  
Each fold guarantees **zero donor overlap** between train / val / test.


In [ ]:
from stagebridge.preprocessing.stage_ontology import normalize_stage_series, CANONICAL_STAGE_ORDER
from stagebridge.training.trainer import (
    build_donor_holdout_splits,
    build_samplers_from_anndata,
    donors_with_min_stage_coverage,
)

STAGE_COL  = "stage"
DONOR_COL  = "patient_id"
N_FOLDS    = 3      # use 5 when ≥10 donors present
SEED       = 42

# Normalise stage labels to canonical ontology
adata_rna_hvg.obs[STAGE_COL] = normalize_stage_series(adata_rna_hvg.obs[STAGE_COL])

obs_stage = np.asarray(adata_rna_hvg.obs[STAGE_COL].astype(str))
obs_donor = np.asarray(adata_rna_hvg.obs[DONOR_COL].astype(str)) if DONOR_COL in adata_rna_hvg.obs.columns \
            else np.array([f"D{i}" for i in range(adata_rna_hvg.n_obs)], dtype=object)

if DONOR_COL not in adata_rna_hvg.obs.columns:
    adata_rna_hvg.obs[DONOR_COL] = obs_donor
    print(f"Warning: '{DONOR_COL}' not found — assigned synthetic donor IDs.")

qualified_donors = donors_with_min_stage_coverage(
    obs_stage=obs_stage,
    obs_donor=obs_donor,
    min_stages=2,
)
print(f"Donors with ≥2 stages: {len(qualified_donors)}  →  {qualified_donors[:10]}")

if len(qualified_donors) < N_FOLDS:
    print(f"[!] Only {len(qualified_donors)} qualified donors — reducing to {max(2, len(qualified_donors))} folds.")
    N_FOLDS = max(2, len(qualified_donors))

splits = build_donor_holdout_splits(donor_ids=qualified_donors, n_folds=N_FOLDS, seed=SEED)

print(f"\nBuilt {N_FOLDS}-fold donor-held-out CV:")
for i, sp in enumerate(splits):
    print(f"  Fold {i}: train={len(sp.train_donors)} donors  "
          f"val={len(sp.val_donors)} donors  "
          f"test={len(sp.test_donors)} donors")

## 15 — Smoke Benchmark Training

Runs **Fold 0** of the StageBridgeModel with a tiny config (2 epochs, CPU) to confirm the full pipeline is wired correctly.  
For a full run use `python scripts/train_stagebridge.py` (with CUDA).

> **Full GPU benchmark**: `python scripts/train_stagebridge.py experiment=full_benchmark`  
> **Smoke CLI run**:       `python scripts/train_stagebridge.py training=smoke model=smoke experiment=smoke`


In [ ]:
import torch
from stagebridge.models.stagebridge import StageBridgeModel
from stagebridge.training.trainer import StageBridgeTrainer
from stagebridge.utils.types import StageBridgeConfig
from stagebridge.utils.seeds import set_global_seed

set_global_seed(SEED)

# ── Resolve device (GPU if available, else CPU) ───────────────────────────
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Training device: {DEVICE}")

# ── Smoke config — tiny model, 2 epochs, CPU-safe ─────────────────────────
SMOKE_DIM = min(N_LATENT_DIM, 16)   # reduce to 16 for speed

smoke_cfg = StageBridgeConfig(
    input_dim=SMOKE_DIM,
    hidden_dim=32,
    vector_field_hidden_dim=64,
    num_heads=2,
    num_inducing_points=4,
    num_seed_vectors=1,
    num_stages=len(CANONICAL_STAGE_ORDER),
    time_embedding_dim=16,
    stage_embedding_dim=16,
    dropout=0.0,
    ot_epsilon=0.05,
    sinkhorn_iters=5,
    num_ot_pairs=64,
    context_consistency_weight=0.1,
    learning_rate=1e-3,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    max_epochs=2,
    steps_per_epoch=2,
    val_steps=1,
    patience=2,
    gradient_accumulation_steps=1,
    mixed_precision=False,
    device=DEVICE,
    seed=SEED,
)

# ── Build a lightweight copy of the latent data (SMOKE_DIM components) ────
import anndata as ad

_lat = np.asarray(adata_rna_hvg.obsm[LATENT_KEY], dtype=np.float32)[:, :SMOKE_DIM]
adata_smoke = ad.AnnData(X=np.zeros((_lat.shape[0], 1), dtype=np.float32))
adata_smoke.obsm[LATENT_KEY]  = _lat
adata_smoke.obs[STAGE_COL]    = adata_rna_hvg.obs[STAGE_COL].values
adata_smoke.obs[DONOR_COL]    = adata_rna_hvg.obs[DONOR_COL].values

# ── Build samplers for Fold 0 ──────────────────────────────────────────────
train_sampler, val_sampler, test_sampler = build_samplers_from_anndata(
    adata=adata_smoke,
    split=splits[0],
    latent_key=LATENT_KEY,
    stage_col=STAGE_COL,
    donor_col=DONOR_COL,
    batch_cells=64,
    device=DEVICE,
)
print(f"Train transitions: {train_sampler.available_transitions}")
print(f"Val   transitions: {val_sampler.available_transitions}")
print(f"Test  transitions: {test_sampler.available_transitions}")

# ── Instantiate model and trainer ─────────────────────────────────────────
smoke_model   = StageBridgeModel(config=smoke_cfg)
smoke_trainer = StageBridgeTrainer(model=smoke_model, config=smoke_cfg)

SMOKE_OUTPUT_DIR = FIGURES_DIR.parent / "smoke_run"

smoke_output = smoke_trainer.fit(
    train_sampler=train_sampler,
    val_sampler=val_sampler,
    test_sampler=test_sampler,
    output_dir=SMOKE_OUTPUT_DIR,
    run_name="smoke",
)

print(f"\nSmoke training complete:")
print(f"  Best val loss   : {smoke_output.best_val_loss:.6f}")
print(f"  Epochs run      : {len(smoke_output.history)}")
print(f"  Checkpoint      : {smoke_output.best_checkpoint}")
print(f"  Benchmark keys  : {list(smoke_output.benchmark_metrics.keys())[:6]}")

## 16 — Evaluation Metrics (per transition)

Computes Sinkhorn distance, MMD-RBF, C2ST-AUC, and composition JSD  
for each adjacent stage transition on the held-out test set of Fold 0.


In [ ]:
from stagebridge.training.eval import evaluate_transition
from stagebridge.preprocessing.stage_ontology import stage_to_index

smoke_model.eval()
eval_rows = []

with torch.no_grad():
    for src_stage, tgt_stage in test_sampler.available_transitions:
        x_src, x_tgt = test_sampler.sample_transition_pair(src_stage, tgt_stage, n_cells=128)
        result = evaluate_transition(
            model=smoke_model,
            x_src=x_src,
            x_tgt=x_tgt,
            stage_src=stage_to_index(src_stage),
            stage_tgt=stage_to_index(tgt_stage),
            num_steps=4,
            ot_epsilon=smoke_cfg.ot_epsilon,
            sinkhorn_iters=smoke_cfg.sinkhorn_iters,
        )
        eval_rows.append({
            "transition":       f"{src_stage} → {tgt_stage}",
            "sinkhorn":         result.sinkhorn,
            "mmd_rbf":          result.mmd_rbf,
            "classifier_auc":   result.classifier_auc,
            "jsd_composition":  result.jsd_composition,
            "rank_consistency": result.rank_consistency,
        })

eval_df = pd.DataFrame(eval_rows)
print("Per-transition evaluation (smoke model, Fold 0 test set):\n")
print(eval_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 17 — Benchmark Summary Figure


In [ ]:
if not eval_df.empty:
    metrics_to_plot = ["sinkhorn", "mmd_rbf", "classifier_auc", "jsd_composition"]
    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 4))

    for ax, metric in zip(axes, metrics_to_plot):
        bars = ax.bar(
            range(len(eval_df)),
            eval_df[metric],
            color="#1f77b4",
            alpha=0.85,
        )
        ax.set_xticks(range(len(eval_df)))
        ax.set_xticklabels(eval_df["transition"], rotation=25, ha="right", fontsize=8)
        ax.set_title(metric.replace("_", " ").title(), fontsize=10)
        ax.grid(alpha=0.2, axis="y")

    fig.suptitle("StageBridge Smoke Benchmark — Fold 0 Test Metrics", fontsize=12, y=1.02)
    fig.tight_layout()
    out = FIGURES_DIR / "benchmark_smoke_metrics.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close(fig)
else:
    print("No eval results — check that test_sampler has available transitions.")

In [ ]:
print("=" * 65)
print("  StageBridge Benchmark — End-to-End Smoke Run Complete")
print("=" * 65)
print()
print(f"  Latent key       : {LATENT_KEY}  ({N_LATENT_DIM}d → smoke {SMOKE_DIM}d)")
print(f"  Donors (CV)      : {len(qualified_donors)}  ({N_FOLDS}-fold donor-held-out)")
print(f"  Model params     : {sum(p.numel() for p in smoke_model.parameters()):,}")
print(f"  Training epochs  : {len(smoke_output.history)}")
print(f"  Best val loss    : {smoke_output.best_val_loss:.6f}")
if smoke_output.benchmark_metrics:
    print(f"  Sinkhorn (mean)  : {smoke_output.benchmark_metrics.get('sinkhorn_mean', float('nan')):.4f}")
    print(f"  MMD-RBF (mean)   : {smoke_output.benchmark_metrics.get('mmd_rbf_mean', float('nan')):.4f}")
    print(f"  C2ST AUC (mean)  : {smoke_output.benchmark_metrics.get('classifier_auc_mean', float('nan')):.4f}")
print()
print("Full GPU run (all models, all folds):")
print("  python scripts/train_stagebridge.py experiment=full_benchmark")
print()
print("Evaluate a saved checkpoint:")
print("  python scripts/eval_stagebridge.py checkpoint=/path/to/checkpoint.pt")
print()
print("Generate poster assets:")
print("  python scripts/make_poster_assets.py outputs/tables/metrics_<run>.json")
print("=" * 65)